# Ancestry Plots

- World Map
- PCA Plot

## 1) World Map

In [7]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Set backend before importing pyplot
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
from cartopy.io.shapereader import Reader
import cartopy.io.shapereader as shpreader
import numpy as np


In [ ]:
superpop_colors = {
    'EUR': '#1976D2',  # Blue
    'AFR': '#FF6F00',  # Orange
    'EAS': '#388E3C',  # Green
    'SAS': '#D32F2F',  # Red
    'AMR': '#7B1FA2'   # Purple
}

# Path to font (Liberation Sans: free, metric-compatible with Helvetica)
font_path = "../../doc/figures/helvetica/LiberationSans-Regular.ttf"

# Register the font
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'Liberation Sans'

# Optional: Set as default for all text
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['font.family'] = 'Liberation Sans'
plt.rcParams['font.size'] = 5
plt.rcParams['axes.titlesize'] = 7
plt.rcParams['axes.labelsize'] = 7
plt.rcParams['xtick.labelsize'] = 5
plt.rcParams['ytick.labelsize'] = 5
plt.rcParams['legend.fontsize'] = 5


In [9]:
# Read the cleaned TSV data
data = pd.read_csv("../../doc/tables/samples_metadata.tsv", sep='\t')

# Add 1000G superpopulation assignments based on country/ethnicity
def assign_superpop(country):
    if country in ['Germany', 'Austria', 'France', 'Spain', 'Croatia', 'Poland', 'Russia']:
        return 'EUR'
    elif country in ['Taiwan', 'Vietnam']:
        return 'EAS'
    elif country in ['India']:
        return 'SAS'
    elif country in ['Mexico', 'Chile']:
        return 'AMR'
    else:
        return 'EUR'  # Default to European

data['superpopulation'] = data['birth_country'].apply(assign_superpop)


In [10]:
# Use only samples from first publication
publication_samples = [
    "T2T00",
    "T2T00.1",
    "T2T00.2",
    "T2T01",
    "T2T02",
    "T2T03",
    "T2T04",
    "T2T04.1",
    "T2T04.2",
    "T2T05",
    "T2T06",
    "T2T07",
    "T2T08",
    "T2T09",
    "T2T10",
    "T2T11",
    "T2T12",
    "T2T13",
    "T2T15",
    "T2T16",
    "T2T17",
    "T2T18",
    "T2T19"
]

data = data[data['t2t_identifier'].isin(publication_samples)]


In [11]:
# Create a text table with counts of birth_country and superpopulations
# Count by birth_country
birth_country_counts = data['birth_country'].value_counts(dropna=False)

# Count by superpopulation
superpop_counts = data['superpopulation'].value_counts(dropna=False)

# Display as text tables
print('Counts by birth_country:')
print(birth_country_counts.to_string())
print('\nCounts by superpopulation:')
print(superpop_counts.to_string())

Counts by birth_country:
birth_country
Germany    7
Russia     2
Austria    1
NaN        1
Spain      1
France     1
Chile      1
India      1
Mexico     1
Taiwan     1
Croatia    1
Vietnam    1

Counts by superpopulation:
superpopulation
EUR    14
AMR     2
EAS     2
SAS     1


In [12]:
# Create the figure with Robinson projection (oval world map)
fig = plt.figure(figsize=(3.5, 2))
ax = plt.axes(projection=ccrs.Robinson())

# Add map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='black')
ax.add_feature(cfeature.BORDERS, linewidth=0.2, color='black', alpha=0.7)
ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.05)
ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.3)

# Set global extent
ax.set_global()

# Get country geometries and records from Natural Earth
shpfilename = shpreader.natural_earth(resolution='50m',
                                       category='cultural',
                                       name='admin_0_countries')
reader = shpreader.Reader(shpfilename)
countries_records = list(reader.records())

# Group data by country and get superpopulation
country_superpop = data.groupby('birth_country')['superpopulation'].first().to_dict()

# Build a set of all Natural Earth country names for validation
ne_country_names = {record.attributes['NAME'] for record in countries_records}

# Check for countries in data that might not match Natural Earth names
for data_country in country_superpop.keys():
    if data_country not in ne_country_names:
        # Check if it's a partial match
        matches = [ne_name for ne_name in ne_country_names if data_country in ne_name or ne_name in data_country]
        if not matches:
            print(f"WARNING: Country '{data_country}' not found in Natural Earth country names")
        else:
            print(f"INFO: Country '{data_country}' matched to: {matches}")

# Shade countries based on superpopulation
for record in countries_records:
    country_name = record.attributes['NAME']
    
    # Check if this country has samples
    for data_country, superpop in country_superpop.items():
        if data_country in country_name or country_name in data_country:
            ax.add_geometries([record.geometry], ccrs.PlateCarree(),
                            facecolor=superpop_colors[superpop],
                            edgecolor='black', linewidth=0.4,
                            alpha=0.9, zorder=5)
            break

# Build explicit legend handles for ALL superpop categories (no filtering)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color, edgecolor='black', label=superpop, alpha=0.9)
    for superpop, color in superpop_colors.items()
]

legend = plt.legend(handles=legend_elements, title='Superpopulation', 
                   loc='lower center', bbox_to_anchor=(0.5, -0.1),
                   edgecolor='black',
                   ncol=len(superpop_colors), frameon=True, fancybox=True)
legend.get_title().set_fontweight('bold')
legend.get_frame().set_alpha(1)
legend.get_frame().set_linewidth(0.4)

# Adjust layout
plt.tight_layout()

# Save the plot
plt.savefig('../../doc/figures/fig2_map.png', dpi=300, bbox_inches='tight', 
           facecolor='white', edgecolor='none')
plt.savefig('../../doc/figures/fig2_map.pdf', bbox_inches='tight', 
           facecolor='white', edgecolor='none')

# Print summary
print("Map saved as '../../doc/figures/fig2_map.png' and '.pdf'")
print("Sample distribution by superpopulation:")
print(data['superpopulation'].value_counts())
print("\nSample distribution by country:")
print(data['birth_country'].value_counts())

plt.show()


/mnt/storage2/users/ahgrosc1/environments/envs/.conda_env/lib/python3.10/site-packages/cartopy/mpl/feature_artist.py:143: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '


Map saved as '../../doc/figures/fig2_map.png' and '.pdf'
Sample distribution by superpopulation:
superpopulation
EUR    14
AMR     2
EAS     2
SAS     1
Name: count, dtype: int64

Sample distribution by country:
birth_country
Germany    7
Russia     2
Austria    1
Spain      1
France     1
Chile      1
India      1
Mexico     1
Taiwan     1
Croatia    1
Vietnam    1
Name: count, dtype: int64


## 2) PCA Plot

In [13]:
# Hardcoded input (PLINK2 eigenvec: FID IID PC1 PC2 ...)
pca_file = "../../analysis_other/ancestry/pca/merge_ref_samples.eigenvec"

# Hardcoded coloring option (SUPERPOP only)
psam_file = "../../analysis_other/ancestry/plink/merge_ref_samples.psam"

# Figure size matches map: (3.5, 2)
fig, ax = plt.subplots(figsize=(3.5, 2))

# Load data
pca = pd.read_csv(pca_file, sep="\t", comment="#", header=None)
pca.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, pca.shape[1]-1)]

psam = pd.read_csv(psam_file, sep="\t", header=0)  # keep header
psam.columns = [c.lstrip("#") for c in psam.columns]  # drop leading '#'
psam = psam[["IID", "SUPERPOP"]]

df = pca.merge(psam, left_on="IID", right_on="IID", how="left")

# SUPERPOP palette (same as map)
superpop_colors = {
    "EUR": "#1976D2",
    "AFR": "#FF6F00",
    "EAS": "#388E3C",
    "SAS": "#D32F2F",
    "AMR": "#7B1FA2",
    "QRY": "#000000",
}

# Finished sample IDs to highlight
highlight_samples = {
    "T2T00", "T2T00_1", "T2T00_2", "T2T04", "T2T04_1", "T2T04_2",
    "T2T01", "T2T02", "T2T03", "T2T05", "T2T06", "T2T07", "T2T08",
    "T2T09", "T2T10", "T2T11", "T2T12", "T2T13", "T2T15", "T2T16",
    "T2T17", "T2T18", "T2T19"
}

# Plot
for sp, color in superpop_colors.items():
    sub = df[df["SUPERPOP"] == sp]
    if not sub.empty:
        # Background points (non-highlight) — circles (outline only)
        bg = sub[~sub["IID"].isin(highlight_samples)]
        if not bg.empty:
            ax.scatter(
                bg["PC1"], bg["PC2"],
                s=6, facecolors="none", edgecolors=color,
                alpha=0.5, linewidths=0.15, marker="o"
            )

        # Highlight points (diamond) — outline only
        hi = sub[sub["IID"].isin(highlight_samples)]
        if not hi.empty:
            ax.scatter(
                hi["PC1"], hi["PC2"],
                s=16, facecolors="none", edgecolors=color, marker="D",
                linewidths=0.25, alpha=0.7
            )

# Build explicit legend handles for ALL superpop categories
import matplotlib.lines as mlines
legend_handles = [
    mlines.Line2D([], [], marker="o", color="none",
                  markerfacecolor="none", markeredgecolor=color,
                  markeredgewidth=0.5, markersize=4, label=sp)
    for sp, color in superpop_colors.items()
]
ax.legend(
    handles=legend_handles,
    title="Superpopulation",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.35),
    ncol=len(superpop_colors),
    frameon=True,
    fancybox=True,
    edgecolor="black",
    fontsize=5,
    title_fontsize=5,
    borderpad=0.4,
    handletextpad=0.3,
    columnspacing=0.6,
).get_title().set_fontweight("bold")

# Styling
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.2)

plt.tight_layout()
plt.savefig("../../doc/figures/fig2_pca_publication.png", dpi=300, bbox_inches="tight")
plt.savefig("../../doc/figures/fig2_pca_publication.pdf", bbox_inches="tight")
plt.show()
